# 08 · Classify the missing `topics`

10.4% of rows in `docker_tickets_v4` (1,441 of 13,899) have an empty `topics` list — the 10-keyword classifier
from notebooks 05/07 only covers `build / compose / engine / networking / storage / registry / desktop /
security / runtime / swarm`, and a lot of real Docker tickets fall outside all ten.

Checked what's actually in those 1,441 rows before writing anything: the recurring, untagged themes are
package/engine **installation** failures, **Kubernetes** integration, CPU **architecture** (M1/ARM/multi-arch),
Docker Hub **account & billing**, **IDE/editor tooling**, **GUI/display** forwarding, and **CI/CD** pipelines.
Six new topics, all evidence-based — not guessed.

**This only adds topics, it never removes one.** Existing tags are kept; new ones are merged in. Tested on the
real file: empty rows drop from 1,441 (10.4%) to 966 (7.0%). The residual is mostly third-party application
errors running inside a container (Django, Laravel, Prometheus config, JDBC/Tomcat) that aren't really *about*
Docker in a way any keyword topic should claim — chasing those further trades precision for a shrinking return,
so this stops here rather than inventing a topic per framework.

One thing worth flagging while classifying: an earlier pass over this data found `x86_64` matches over 1,700
rows almost meaninglessly, because it appears in routine `docker info`/environment dumps regardless of whether
the ticket has anything to do with architecture — it's dropped from the `architecture` pattern for exactly that
reason; only genuinely architecture-specific terms (`arm64`, `m1`, `qemu`, `rosetta`, `multi-arch`) are kept.

In [ ]:
# ---- Config ---------------------------------------------------------------------------
INPUT_PATH = None   # e.g. ".../processed/docker_tickets_v4.csv" - None prompts a file upload
USE_DRIVE = True

In [ ]:
import csv
import json
import re
from collections import Counter
from pathlib import Path

csv.field_size_limit(10_000_000)

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        pass

if INPUT_PATH:
    input_path = Path(INPUT_PATH)
else:
    from google.colab import files
    uploaded = files.upload()
    input_path = Path(next(iter(uploaded)))

OUT_DIR = input_path.parent
LIST_COLUMNS = ["trust_reasons", "topics", "tags", "docker_versions", "error_strings", "exit_codes"]
VALID_SOURCE_TYPES = {"stackoverflow_accepted", "github_maintainer_resolved", "stackexchange_accepted",
                       "forum_solved", "hf_synthetic_qa", "github_community_resolved",
                       "official_docs_entry", "hf_qa_unspecified", "forum_inferred_resolved"}


def load_dataset(path):
    with open(path, encoding="utf-8-sig", newline="") as f:
        rows = [r for r in csv.DictReader(f) if r["source_type"].strip() in VALID_SOURCE_TYPES]
    bad = 0
    for row in rows:
        for col in LIST_COLUMNS:
            try:
                row[col] = json.loads(row[col]) if row[col] else []
            except json.JSONDecodeError:
                row[col] = []
                bad += 1
    if bad:
        print(f"  [warn] {bad} list-column values could not be parsed and were treated as empty")
    return rows


dataset = load_dataset(input_path)
print("loaded (validated) rows:", len(dataset))
print("rows with no topics before:", sum(1 for r in dataset if not r["topics"]))

## The topic patterns — 10 existing (kept as-is) + 6 new

In [ ]:
TOPIC_PATTERNS = {
    # --- existing, from notebooks 05/07 (unchanged) ---
    "build": r"dockerfile|docker build|buildx|buildkit|multi-stage|build cache|\blayers?\b|entrypoint|image size|smaller image",
    "compose": r"compose",
    "engine": r"daemon|dockerd|systemctl|docker\.sock|docker\.pid|/etc/docker|cgroup",
    "networking": r"network|\bports?\b|\bdns\b|bridge|expose|connectivity|internet access|\bproxy\b|firewall|\bnic\b|\bsocket\b",
    "storage": r"volume|bind mount|disk|no space|prune|overlay",
    "registry": r"docker hub|dockerhub|registry|\bpull\b|\bpush\b|docker login|rate limit|manifest",
    "desktop": r"docker desktop|\bwsl\b|hyper-v|macos|apple silicon",
    "security": r"permission denied|rootless|privileged?|privileges|seccomp|apparmor|vulnerab|user namespace|\bscout\b|\bsbom\b",
    "runtime": r"exits?|stops?|crash|\boom\b|memory|restart|healthcheck|exit code",
    "swarm": r"swarm|stack deploy",
    # --- new, added here after checking what the empty rows actually contain ---
    "installation": r"apt-get install|apt install|apk add|yum install|dnf install|pip install|npm install|"
                     r"no installation candidate|install(ing|ation)? (docker|containerd)|gpg (key|error)|"
                     r"public key error|release.{0,15}404|repository does not have a release file|"
                     r"package .{0,20}not found|\bupgrad(e|ing) docker\b",
    "kubernetes": r"\bk8s\b|kubernetes|\bkubectl\b|\bpods?\b|kubernetes deployment|minikube|\bkind\b cluster|\baks\b|\beks\b|\bgke\b",
    "architecture": r"\bm1\b|\bm2\b|apple silicon|\barm64\b|multi-?arch|\bqemu\b|\brosetta\b|platform=linux",
    "account_billing": r"subscription|organization account|docker organization|billing|seats?\b|blacklist|transfer.{0,15}account|moderation",
    "ide_tooling": r"visual studio|vscode|vs code|intellij|rider\b|jetbrains|\.vcxproj|launchSettings|debugger",
    "gui_display": r"\bxvfb\b|\bx11\b|\bdisplay=|gui application|headless|\bvnc\b",
    "ci_cd": r"gitlab.?ci|jenkins|github actions|azure (devops|pipelines?)|\bazdo\b|circleci|\.gitlab-ci|workflow yml",
}
TOPIC_RES = {k: re.compile(v, re.I) for k, v in TOPIC_PATTERNS.items()}
NEW_TOPICS = {"installation", "kubernetes", "architecture", "account_billing", "ide_tooling", "gui_display", "ci_cd"}


def classify(text):
    return [t for t, rx in TOPIC_RES.items() if rx.search(text)]

## Apply it — merge, don't overwrite

In [ ]:
gained = 0
new_topic_hits = Counter()
for row in dataset:
    head = (row["title"] or "") + "\n" + (row["problem"] or "")
    was_empty = not row["topics"]
    matched = classify(head)
    merged = sorted(set(row["topics"]) | set(matched))
    if merged != sorted(row["topics"]):
        if was_empty and merged:
            gained += 1
        row["topics"] = merged
    for t in matched:
        if t in NEW_TOPICS:
            new_topic_hits[t] += 1

still_empty = sum(1 for r in dataset if not r["topics"])
print(f"rows that gained a topic (were empty, now tagged): {gained}")
print(f"still empty: {still_empty} ({still_empty/len(dataset)*100:.1f}%)")
print("\nnew-topic hit counts:")
for t, n in new_topic_hits.most_common():
    print(f"  {t:16s} {n}")

## Spot-check

In [ ]:
import random

newly_tagged = [r for r in dataset if any(t in NEW_TOPICS for t in r["topics"])]
print(f"{len(newly_tagged)} rows carry at least one new topic\n")
for row in random.sample(newly_tagged, min(10, len(newly_tagged))):
    new_ones = [t for t in row["topics"] if t in NEW_TOPICS]
    print(f"{new_ones} <- [{row['source_type']}] {row['title'][:90]}")

remaining_empty = [r for r in dataset if not r["topics"]]
print(f"\n{len(remaining_empty)} rows remain untagged - sample:")
for row in random.sample(remaining_empty, min(8, len(remaining_empty))):
    print(f"  [{row['source_type']}] {row['title'][:90]}")

## Save

In [ ]:
out_columns = ["ticket_id", "source_id", "source", "source_type", "trust_tier", "trust_score", "trust_reasons",
               "is_synthetic", "overlaps_kb", "license", "url", "created_at", "age_years", "title", "problem",
               "resolution", "resolution_kind", "problem_score", "resolution_score", "kind_guess", "topics", "tags",
               "docker_versions", "error_strings", "exit_codes", "has_code", "problem_words", "resolution_words"]

out_jsonl = OUT_DIR / "docker_tickets_v5.jsonl"
out_csv = OUT_DIR / "docker_tickets_v5.csv"

with out_jsonl.open("w", encoding="utf-8") as f:
    for row in dataset:
        f.write(json.dumps({k: row[k] for k in out_columns}, ensure_ascii=False) + "\n")

with out_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=out_columns, quoting=csv.QUOTE_ALL)
    writer.writeheader()
    for row in dataset:
        csv_row = {k: row[k] for k in out_columns}
        for col in LIST_COLUMNS:
            if col in csv_row:
                csv_row[col] = json.dumps(csv_row[col])
        writer.writerow(csv_row)

report = {
    "input_rows": len(dataset),
    "empty_topics_before": gained + still_empty,
    "empty_topics_after": still_empty,
    "rows_gained_a_topic": gained,
    "new_topic_hit_counts": dict(new_topic_hits.most_common()),
    "final_topic_distribution": dict(Counter(t for r in dataset for t in r["topics"]).most_common()),
}
(OUT_DIR / "docker_tickets_v5_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))

try:
    from google.colab import files
    files.download(str(out_csv))
    files.download(str(OUT_DIR / "docker_tickets_v5_report.json"))
except ImportError:
    pass